## ENEM 2024 — Agregações nacionais para dashboard

Objetivo: gerar tabelas agregadas em nível Brasil a partir das bases tratadas de participantes e resultados do ENEM 2024, produzindo saídas compactas e adequadas para visualizações, comparações entre grupos e etapas posteriores de consolidação.

Entradas:

* DADOS_TRATADOS_MEDIAS_2024 — base tratada de participantes, com variáveis socioeconômicas e derivadas
* RESULTADOS_TRATADOS_MEDIAS_2024 — base tratada de resultados, com notas e indicadores de presença

Saídas:

* DADOS_AGG_2024 — agregação socioeconômica por grupos
* RESULTADOS_AGG_2024 — agregação de notas e presença por grupo
* MERGED_2024 — união entre indicadores socioeconômicos e desempenho
* AMOSTRAG_RESULTADOS_2024 — amostra estratificada por percentis de nota_media

Observação metodológica: em 2024, os microdados de participantes e resultados foram divulgados em arquivos separados e não podem ser associados em nível individual. Por isso, a integração entre perfil socioeconômico e desempenho é feita apenas em nível agregado.

Observação (GitHub): esta etapa opera exclusivamente sobre arquivos Parquet já tratados nas etapas anteriores. Os microdados brutos não são versionados no repositório.



### 1) Ambiente e imports

In [1]:
import sys
from pathlib import Path

ROOT_PATH = Path().resolve().parents[1]
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import pandas as pd

from src.config import (
    DADOS_TRATADOS_MEDIAS_2024,
    RESULTADOS_TRATADOS_MEDIAS_2024,
    AMOSTRAG_RESULTADOS_2024,
    DADOS_AGG_2024,
    RESULTADOS_AGG_2024,
    MERGED_2024,
)

from src.preprocessamento.agregacoes import  (
    agrupar_notas, 
    amostrar_por_percentil_original,
    categoria_ordenada_para_numero,
)



pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")



### 2) Leitura das bases tratadas

Nesta etapa são carregadas separadamente as bases tratadas de participantes e de resultados do ENEM 2024. Como os arquivos originais de 2024 não permitem associação direta em nível individual, as análises conjuntas são realizadas por meio de agregações compatíveis entre grupos.

In [2]:
df_d = pd.read_parquet(DADOS_TRATADOS_MEDIAS_2024)
df_r = pd.read_parquet(RESULTADOS_TRATADOS_MEDIAS_2024)
df_d.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,municipio,uf,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,comptdr,cel,escola,indice_consumo,renda_media
0,20-25,feminino,solteiro,branca,Porto Alegre,RS,superior,superior,manual/tec,médio/tec,4,1 a 3,0,1,1,3,1,4,pública,0.66,2.00
1,26-35,feminino,solteiro,branca,São Luiz Gonzaga,RS,até fund,até fund,rural,médio/tec,2,1 a 3,0,1,0,1,0,2,pública,0.25,2.00
2,26-35,feminino,solteiro,branca,Sarandi,RS,desconhecida,superior,desconhecida,médio/tec,5,5 a 10,0,2,1,3,0,3,pública,0.70,7.50


In [3]:
df_d['escola'].unique()

['pública', 'privada', 'não informada']
Categories (3, object): ['pública' < 'não informada' < 'privada']

In [4]:
df_r.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4332944 entries, 0 to 4332943
Data columns (total 10 columns):
 #   Column        Dtype   
---  ------        -----   
 0   uf            category
 1   escola        category
 2   municipio     category
 3   nota_cn       float32 
 4   nota_ch       float32 
 5   nota_lc       float32 
 6   nota_mt       float32 
 7   lingua        category
 8   nota_redacao  float32 
 9   nota_media    float32 
dtypes: category(4), float32(6)
memory usage: 119.9 MB


In [5]:
df_r.head(3)

,uf,escola,municipio,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,nota_media
0,CE,pública,Aratuba,436.80,377.80,423.40,427.10,espanhol,300.00,393.02
1,SC,privada,Tijucas,521.90,601.90,605.50,689.20,inglês,920.00,667.70
2,PR,não informada,Rolândia,363.00,548.40,557.20,456.40,espanhol,480.00,481.00


### 3) Conversão para códigos numéricos

In [6]:
df_d_num = df_d.copy()


for col in ["ocup_pai", "ocup_mae", "escolaridade_pai", "escolaridade_mae"]:
    df_d_num[col] = categoria_ordenada_para_numero(df_d_num[col])
df_d_num['escola_num']=categoria_ordenada_para_numero(df_d_num['escola'])


df_d_num['escolaridade_pais_media'] = (df_d_num['escolaridade_pai'] + df_d_num['escolaridade_mae']) / 2
df_d_num['ocup_pais_media'] = (df_d_num['ocup_mae'] + df_d_num['ocup_pai']) / 2
df_d_num.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,municipio,uf,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,comptdr,cel,escola,indice_consumo,renda_media,escola_num,escolaridade_pais_media,ocup_pais_media
0,20-25,feminino,solteiro,branca,Porto Alegre,RS,4,4,3,4,4,1 a 3,0,1,1,3,1,4,pública,0.66,2.00,0,4.00,3.50
1,26-35,feminino,solteiro,branca,São Luiz Gonzaga,RS,2,2,1,4,2,1 a 3,0,1,0,1,0,2,pública,0.25,2.00,0,2.00,2.50
2,26-35,feminino,solteiro,branca,Sarandi,RS,0,4,0,4,5,5 a 10,0,2,1,3,0,3,pública,0.70,7.50,0,2.00,2.00


### 4) Agregação da base de participantes

A base de participantes é agregada por um conjunto de variáveis demográficas e socioeconômicas. O objetivo é produzir perfis médios de grupos populacionais, preservando:

* tamanho do grupo (participantes)
* acesso a bens e infraestrutura (cel, comptdr, indice_consumo)
* estrutura domiciliar (n_pessoas_resd)
* renda média aproximada (renda_media)

Essas agregações alimentam o dashboard e permitem posterior integração com indicadores de desempenho em nível agregado.

In [7]:
for base in [df_d, df_r, df_d_num]:
    base['ano']='2024'
    base["ano"] = base["ano"].astype(str)
   

In [8]:
cols_agg = [
    "ano",
    "uf",
    "escola",
    "sexo",
    "cor_raca",
    "faixa_etaria",
    "sal_min",
    "escolaridade_pai",
    "escolaridade_mae",
    "ocup_pai",
    "ocup_mae",
]

for col in cols_agg:
    if df_d[col].dtype.name == "category":
        df_d[col] = df_d[col].cat.remove_unused_categories()

df_dados_agg = (
    df_d.groupby(cols_agg, observed=True, dropna=False)
    .agg(
        participantes=("uf", "count"),
        cel=("cel", "mean"),
        comptdr=("comptdr", "mean"),
        n_pessoas_resd=("n_pessoas_resd", "mean"),
        renda_media=("renda_media", "mean"),
        indice_consumo=("indice_consumo", "mean"),
    )
    .reset_index()
)

df_dados_agg = df_dados_agg[df_dados_agg["participantes"] > 0]

df_dados_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600309 entries, 0 to 600308
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   ano               600309 non-null  object  
 1   uf                600309 non-null  category
 2   escola            600309 non-null  category
 3   sexo              600309 non-null  category
 4   cor_raca          600309 non-null  category
 5   faixa_etaria      600309 non-null  category
 6   sal_min           600309 non-null  category
 7   escolaridade_pai  600309 non-null  category
 8   escolaridade_mae  600309 non-null  category
 9   ocup_pai          600309 non-null  category
 10  ocup_mae          600309 non-null  category
 11  participantes     600309 non-null  int64   
 12  cel               600309 non-null  Float64 
 13  comptdr           600309 non-null  Float64 
 14  n_pessoas_resd    600309 non-null  Float64 
 15  renda_media       600309 non-null  float32 
 16  in

### 5) Agregação da base de resultados

A base de resultados é agregada para sintetizar o desempenho médio e máximo dos grupos, bem como indicadores de presença e ausência nos dias de prova. Essa etapa permite resumir o comportamento educacional dos grupos sem depender de identificação individual entre as bases.

São calculados:

* médias das notas por área e da nota_media
* maiores notas observadas por área
* desvio-padrão da nota_media
* quantidade de inscritos
* faltosos e taxas de presença por dia de prova



In [9]:
df_resultados_agg = agrupar_notas(df_r)
df_resultados_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   ano                 81 non-null     object  
 1   uf                  81 non-null     category
 2   escola              81 non-null     category
 3   nota_media          81 non-null     float32 
 4   nota_cn             81 non-null     float32 
 5   nota_cn_max         81 non-null     float32 
 6   nota_ch             81 non-null     float32 
 7   nota_ch_max         81 non-null     float32 
 8   nota_lc             81 non-null     float32 
 9   nota_lc_max         81 non-null     float32 
 10  nota_mt             81 non-null     float32 
 11  nota_mt_max         81 non-null     float32 
 12  nota_redacao        81 non-null     float32 
 13  nota_redacao_max    81 non-null     float32 
 14  desvio_padrao       81 non-null     float32 
 15  participantes       81 non-null     int64 

### 6) Amostragem estratificada por percentis


Além das tabelas agregadas, foi criada uma amostra estratificada da base de resultados, distribuída de forma aproximadamente uniforme ao longo dos percentis de nota_media. O objetivo é preservar a diversidade do desempenho observado e produzir uma amostra mais leve para explorações visuais e análises complementares.

In [10]:
df_amostra_resultados = amostrar_por_percentil_original(df_r, escopo='br', q=20, n_por_percentil=150)
print(f"Amostra com distribuição original: {len(df_amostra_resultados)} linhas")

 

Amostra com distribuição original: 230662 linhas


In [11]:
df_amostra_resultados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230662 entries, 0 to 230661
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   uf            230662 non-null  category
 1   escola        230662 non-null  category
 2   municipio     230662 non-null  category
 3   nota_cn       219453 non-null  float32 
 4   nota_ch       229781 non-null  float32 
 5   nota_lc       229781 non-null  float32 
 6   nota_mt       219453 non-null  float32 
 7   lingua        230662 non-null  category
 8   nota_redacao  229781 non-null  float32 
 9   nota_media    230662 non-null  float32 
 10  ano           230662 non-null  object  
dtypes: category(4), float32(6), object(1)
memory usage: 8.2+ MB


### 7) Merge entre dados agregados e resultados agregados


Como os microdados de participantes e resultados de 2024 não podem ser associados individualmente, a integração entre perfil socioeconômico e desempenho é realizada em nível agregado. Para isso, é construída uma base intermediária resumida por uf e escola, posteriormente unida às estatísticas de notas e presença calculadas na base de resultados.

In [12]:
cols_merged = ["ano", "uf", "escola"]

df_d_merged = (
    df_d_num.groupby(cols_merged, observed=True, dropna=False)
    .agg(
        renda_media=("renda_media", "mean"),
        participantes=("sexo", "count"),
        cel=("cel", lambda x: round(x.mean())),
        comptdr=("comptdr", lambda x: round(x.mean())),
        n_pessoas_resd=("n_pessoas_resd", lambda x: round(x.mean())),
        indice_consumo=("indice_consumo", "mean"),
        escolaridade_pais_media=('escolaridade_pais_media','mean'),
        ocup_pais_media=('ocup_pais_media','mean'),
        escola_num=('escola_num','mean')
    )
    .reset_index()
)

df_d_merged = df_d_merged[df_d_merged["participantes"] > 0]


df_merged = pd.merge(df_d_merged, df_resultados_agg, on=["ano", "uf", "escola"], how="inner")

df_merged = df_merged.drop("participantes_y", axis=1)
df_merged = df_merged.rename(columns={"participantes_x": "participantes"})

df_merged.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   ano                      81 non-null     object  
 1   uf                       81 non-null     category
 2   escola                   81 non-null     category
 3   renda_media              81 non-null     float32 
 4   participantes            81 non-null     int64   
 5   cel                      81 non-null     Int8    
 6   comptdr                  81 non-null     Int8    
 7   n_pessoas_resd           81 non-null     Int8    
 8   indice_consumo           81 non-null     float32 
 9   escolaridade_pais_media  81 non-null     Float64 
 10  ocup_pais_media          81 non-null     Float64 
 11  escola_num               81 non-null     Float64 
 12  nota_media               81 non-null     float32 
 13  nota_cn                  81 non-null     float32 
 14  nota_cn_max 

### 8) Exportação


Ao final desta etapa, são salvos quatro arquivos:

* agregação socioeconômica da base de participantes;
* agregação de desempenho e presença da base de resultados;
* base agregada unificada para análises comparativas;
* amostra estratificada da base de resultados.

Esses arquivos serão utilizados nas etapas posteriores de dashboard e consolidação.

In [13]:
df_dados_agg.to_parquet(DADOS_AGG_2024, index=False)
df_resultados_agg.to_parquet(RESULTADOS_AGG_2024, index=False)
df_merged.to_parquet(MERGED_2024, index=False)
df_amostra_resultados.to_parquet(AMOSTRAG_RESULTADOS_2024, index=False)

print("✅ Salvo base dados agregados 2024:", DADOS_AGG_2024, "| shape:", df_dados_agg.shape)
print("✅ Salvo base resultados agregados 2024:", RESULTADOS_AGG_2024, "| shape:", df_resultados_agg.shape)
print("✅ Salvo dataframe merged BR 2024:", MERGED_2024, "| shape:", df_merged.shape)
print("✅ Salvo amostra dos resultados 2024:", AMOSTRAG_RESULTADOS_2024, "| shape:", df_amostra_resultados.shape)

✅ Salvo base dados agregados 2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\dados_agg_24.parquet | shape: (600309, 17)
✅ Salvo base resultados agregados 2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\resultado_agg_24.parquet | shape: (81, 23)
✅ Salvo dataframe merged BR 2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\merged_24.parquet | shape: (81, 31)
✅ Salvo amostra dos resultados 2024: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\amostrag_24.parquet | shape: (230662, 11)
